# Paper 5, RB-1 - ViT-S drift check

**Naming.** W1-W7 are the September runs behind the submitted paper. Everything run in November
for the rebuttal is labelled **RB**. RB-0 is the single-head run finished on 2 Nov (its files
are named `w8_*`). This notebook is **RB-1** and writes to `./rb1_output/`.

**Why this run exists.** RB-0 failed 4 of 233 gates, all on naver ViT-S. Its SSDC values came out up
to 4.7e-4 away from W5 (13 Sept). The other four models moved by at most 1.1e-6. In September every
run was bit-identical to every other (W2-W6, 10-14 Sept, naver max |difference| = 0.0), so something in
the computing environment changed between 14 Sept and 2 Nov. ViT-S's difference is already present at
the input to block 0, before any rotation or intervention acts, so the change sits in the image →
patch-embedding step (images, the patch-embedding convolution, or the weights).

**What RB-1 does.** It changes no paper result and runs no new intervention. It re-measures three
conditions that the paper already reports (baseline, all rotations off = floor, rotations off in
blocks 0-5 = Cref) with **W5's exact code path**, under several settings, each in its own fresh Python
process:

| Setting | What changes | What a match with September would mean |
|---|---|---|
| `default_a`, `default_b` | nothing (as RB-0), run twice | current environment is repeatable |
| `w8_order` | models loaded naver → DINOv3 → ViT-S, the order W5 and RB-0 used | kernel choice depends on what ran before in the process |
| `tf32_off` | patch-embedding convolution in true fp32 (cuDNN TF32 off) | September did not use TF32 |
| `cudnn_deterministic` | cuDNN restricted to deterministic kernels | a non-deterministic kernel was picked |
| `hog_58GB` | a second process holds ~58 GB of GPU memory, as the September co-tenant job did | **free GPU memory changed which convolution kernel was used** |
| `exact_patch` | patch embedding computed in float64, then rounded to fp32 | reference: how far TF32 rounding moves each model |
| `gpu_k` | default on every other visible GPU | a different physical GPU was used |

It also records forensics: package versions and install dates, driver, cuDNN, GPU processes, and the
Hugging Face commit history of the weights and the image dataset.

**TF32** is a GPU arithmetic mode that keeps the fp32 range but only ~3 significant digits. PyTorch
uses it for convolutions by default (`torch.backends.cudnn.allow_tf32 = True`; RB-0 printed this).
Different cuDNN kernels round differently in TF32, so the choice of kernel can move the patch
embeddings slightly. That is the leading hypothesis.

Runtime: ~20-25 min on one H100. Each setting saves when it finishes.

In [1]:
import importlib, subprocess, sys
def ensure(p, m=None):
    try:
        importlib.import_module(m or p); return False
    except ImportError:
        print(f"installing {p} ..."); subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p]); return True
for p in ["timm", "datasets", "scipy", "pandas", "huggingface_hub"]:
    ensure(p)
print("environment ready")

/workspace/NRDL/env_paper5/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


environment ready


In [2]:
# this parent process never touches the GPU (all GPU work runs in subprocesses), so it cannot change
# the free memory, the cuDNN plan cache or anything else the settings below test
import os, json, time, shutil, signal, hashlib, glob, re, platform, traceback
from datetime import datetime, timezone
from pathlib import Path
import numpy as np

# config
REPO_PATH   = os.environ.get("RB1_REPO", "")        # .../vit-sae-analysis/project_code/src ("" = search)
W8_DIR      = os.environ.get("RB1_W8_DIR", "")      # RB-0 output folder (w8_output); optional, for a cross-check
OUT_DIR     = Path(os.environ.get("RB1_OUT", "rb1_output"))
TEST_MODE   = os.environ.get("RB1_TEST", "0") == "1"   # CPU smoke test on tiny random models, not for results

HOG_GB          = float(os.environ.get("RB1_HOG_GB", "58"))   # September co-tenant footprint (W3 note: 58 of 81 GB)
EXTRA_GPU_RUNS  = os.environ.get("RB1_EXTRA_GPUS", "1") == "1"
CUDNN_LOGGING   = os.environ.get("RB1_CUDNN_LOG", "1") == "1"
ONLY            = [c for c in os.environ.get("RB1_ONLY", "").split(",") if c]   # rerun named settings only
CUTOFF          = "2026-09-14T00:00:00+00:00"   # last bit-identical September run (W6) was 14 Sept

# W5's protocol, unchanged
N_IMAGES, BATCH_SIZE, TORCH_SEED, METRIC, INPUT_SIZE = 1000, 256, 20260908, "manhattan", 224
SSDC_DATASET = "benjamin-paine/imagenet-1k-256x256"
WINDOW = list(range(0, 6))
MODEL_NAMES = {"small":  "vit_small_patch16_rope_224.naver_in1k",
               "naver":  "vit_base_patch16_rope_224.naver_in1k",
               "DINOv3": "vit_base_patch16_dinov3.lvd1689m"}
CONDITIONS = ["baseline", "floor", "Cref"]

SETTINGS = [   # name, flags, models in load order
    ("default_a",           {},                               ["small", "naver"]),
    ("default_b",           {},                               ["small", "naver"]),
    ("w8_order",            {},                               ["naver", "DINOv3", "small"]),
    ("tf32_off",            {"cudnn_allow_tf32": False},      ["small", "naver"]),
    ("cudnn_deterministic", {"cudnn_deterministic": True},    ["small", "naver"]),
    (f"hog_{int(HOG_GB)}GB", {"hog_gb": HOG_GB},              ["small", "naver"]),
    ("exact_patch",         {"exact_patch": True},            ["small", "naver"]),
]
if TEST_MODE:
    N_IMAGES, BATCH_SIZE = 24, 8
    SETTINGS = [s for s in SETTINGS if not s[0].startswith("hog")]
    OUT_DIR = Path(os.environ.get("RB1_OUT", "rb1_test_output"))
OUT_DIR.mkdir(exist_ok=True, parents=True)

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H-%M-%SZ")
GATES = []
GATE_FILE = OUT_DIR / f"rb1_gates_session_{RUN_STAMP}.json"
def gate(name, scope, passed, detail="", skip=False):
    status = "SKIP" if skip else ("PASS" if passed else "FAIL")
    GATES.append({"gate": name, "scope": scope, "status": status, "detail": detail, "session": RUN_STAMP})
    print(f"  [{status}] {scope}: {name}" + (f": {detail}" if detail else ""), flush=True)
    GATE_FILE.write_text(json.dumps(GATES, indent=1))
    return passed
print(f"settings: {[s[0] for s in SETTINGS]} | images {N_IMAGES} | batch {BATCH_SIZE} | test mode {TEST_MODE}")
print(f"output {OUT_DIR.resolve()} | run {RUN_STAMP}")

settings: ['default_a', 'default_b', 'w8_order', 'tf32_off', 'cudnn_deterministic', 'hog_27GB', 'exact_patch'] | images 1000 | batch 256 | test mode False
output /workspace/NRDL/experiments/rb1/rb1_output | run 2026-11-04T00-06-44Z


In [3]:
def resolve_repo():
    cands = []
    if REPO_PATH:
        cands.append(Path(REPO_PATH).expanduser())
    here = Path.cwd()
    for base in [here, *list(here.parents)[:4]]:
        cands += [base / "vit-sae-analysis" / "project_code" / "src", base / "project_code" / "src", base / "src", base]
    for c in cands:
        c = c.resolve()
        if (c / "metrics").exists() and (c / "main").exists() and (c / "experiments").exists():
            return c
    raise FileNotFoundError("No checkout with metrics/, main/, experiments/. Set RB1_REPO.")

repo = resolve_repo()
print("repo root:", repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from metrics.ssdc import spatial_similarity_distance_correlation   # CPU only
from scipy.spatial.distance import cdist as _cdist
_G = 14
_c = np.stack(np.meshgrid(np.arange(_G), np.arange(_G), indexing="ij"), -1).reshape(-1, 2)
_D = _cdist(_c, _c, metric="cityblock")
_a = spatial_similarity_distance_correlation(-_D, grid_size=_G, metric="manhattan")
_b = spatial_similarity_distance_correlation(_D, grid_size=_G, metric="manhattan")
gate("imported metric satisfies its unit test", "repo-metric", _a > 0.999 and _b < -0.999, f"{_a:+.3f}/{_b:+.3f}")
try:
    _head = subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
except Exception:
    _head = ""
print("repo commit:", _head or "(unknown)")

repo root: /workspace/NRDL/clone/vit-sae-analysis/project_code/src


  [PASS] repo-metric: imported metric satisfies its unit test: +1.000/-1.000


repo commit: 28e0856cd87c8582aff2783368b3d2f098ae3190


## Part 0 - forensics (no GPU)
What changed on this machine since 14 Sept? Packages (version and install time), NVIDIA driver,
processes on the GPU, the local Hugging Face cache, and the commit history of the two checkpoints and
the image dataset on the Hub. Everything is best-effort: a check that cannot run is recorded, not fatal.

In [4]:
def sh(cmd, timeout=60):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return (r.stdout or "") + (r.stderr or "")
    except Exception as ex:
        return f"(failed: {ex!r})"

def iso(ts): return datetime.fromtimestamp(ts, timezone.utc).isoformat()

FOR = {"generated_utc": RUN_STAMP, "cutoff": CUTOFF, "python": sys.version, "platform": platform.platform(),
       "executable": sys.executable, "repo_commit": _head}

# packages
import importlib.metadata as md
KEY_PKGS = ["torch", "torchvision", "timm", "pillow", "pillow-simd", "numpy", "scipy", "datasets",
            "huggingface-hub", "safetensors", "transformers", "pyarrow", "fsspec",
            "nvidia-cudnn-cu12", "nvidia-cublas-cu12", "nvidia-cuda-runtime-cu12", "nvidia-cuda-nvrtc-cu12",
            "nvidia-cufft-cu12", "nvidia-nvjitlink-cu12", "triton"]
pk = []
for d in md.distributions():
    name = (d.metadata.get("Name") or "").lower()
    path = getattr(d, "_path", None)
    mt = None
    if path is not None:
        try:
            rec = Path(path) / "RECORD"
            mt = max(os.stat(path).st_mtime, os.stat(rec).st_mtime if rec.exists() else 0)
        except Exception:
            pass
    pk.append({"name": name, "version": d.version, "installed_utc": iso(mt) if mt else None})
pk.sort(key=lambda r: r["name"])
cut = datetime.fromisoformat(CUTOFF)
FOR["packages_key"] = [r for r in pk if r["name"] in KEY_PKGS]
FOR["packages_changed_after_cutoff"] = [r for r in pk if r["installed_utc"] and datetime.fromisoformat(r["installed_utc"]) > cut]
(OUT_DIR / "rb1_pip_freeze.txt").write_text(sh(f"{sys.executable} -m pip freeze"))
print("key packages:")
for r in FOR["packages_key"]:
    flag = "  <-- installed/changed after 14 Sept" if r["installed_utc"] and datetime.fromisoformat(r["installed_utc"]) > cut else ""
    print(f"  {r['name']:26s} {r['version']:18s} {r['installed_utc'] or '?'}{flag}")
print(f"{len(FOR['packages_changed_after_cutoff'])} packages in this environment were installed or changed after 14 Sept")
for r in FOR["packages_changed_after_cutoff"][:40]:
    print(f"    {r['name']:30s} {r['version']:16s} {r['installed_utc']}")

# driver and GPUs
FOR["nvidia_smi_gpus"] = sh("nvidia-smi --query-gpu=index,name,uuid,driver_version,pci.bus_id,memory.total,memory.used --format=csv")
FOR["nvidia_smi_apps"] = sh("nvidia-smi --query-compute-apps=gpu_uuid,pid,process_name,used_memory --format=csv")
FOR["nvidia_proc_version"] = sh("cat /proc/driver/nvidia/version")
FOR["libcuda_files"] = sh("ls -l --time-style=full-iso /usr/lib/x86_64-linux-gnu/libcuda.so* /usr/lib64/libcuda.so* 2>/dev/null")
FOR["driver_package_log"] = sh("(zgrep -h -i 'nvidia' /var/log/dpkg.log* 2>/dev/null | grep -E ' (install|upgrade) ' | tail -40) ; "
                               "(zgrep -h -i -B1 -A3 'nvidia' /var/log/apt/history.log* 2>/dev/null | tail -60)")
FOR["uptime_since"] = sh("uptime -s")
print("\nGPUs:\n" + FOR["nvidia_smi_gpus"])
print("processes on the GPUs now:\n" + FOR["nvidia_smi_apps"])
print("driver:", FOR["nvidia_proc_version"].strip().splitlines()[0] if FOR["nvidia_proc_version"].strip() else "(unknown)")
print("machine up since:", FOR["uptime_since"].strip())
GPU_INDICES = [int(m.group(1)) for m in re.finditer(r"^GPU (\d+):", sh("nvidia-smi -L"), re.M)]
print("visible GPU indices:", GPU_INDICES)

# Hugging Face: local cache and remote commit history
HF_REPOS = [("timm/" + MODEL_NAMES["small"], "model"), ("timm/" + MODEL_NAMES["naver"], "model"),
            ("timm/" + MODEL_NAMES["DINOv3"], "model"), (SSDC_DATASET, "dataset")]
FOR["hf_cache"], FOR["hf_remote"] = {}, {}
try:
    from huggingface_hub import scan_cache_dir, HfApi
    info = scan_cache_dir()
    for r in info.repos:
        if any(r.repo_id == rid for rid, _ in HF_REPOS):
            FOR["hf_cache"][r.repo_id] = [{"commit": rev.commit_hash, "refs": sorted(rev.refs),
                                           "last_modified_utc": iso(rev.last_modified),
                                           "files": [{"name": f.file_name, "size": f.size_on_disk,
                                                      "blob_modified_utc": iso(f.blob_last_modified)} for f in rev.files]}
                                          for rev in r.revisions]
    api = HfApi()
    for rid, rtype in HF_REPOS:
        try:
            commits = api.list_repo_commits(rid, repo_type=rtype)
            FOR["hf_remote"][rid] = [{"commit": c.commit_id, "date": c.created_at.isoformat(), "title": c.title} for c in commits[:15]]
        except Exception as ex:
            FOR["hf_remote"][rid] = {"error": repr(ex)[:200]}
except Exception as ex:
    FOR["hf_error"] = repr(ex)[:300]
print()
for rid, _ in HF_REPOS:
    loc = FOR["hf_cache"].get(rid, [])
    rem = FOR["hf_remote"].get(rid, [])
    late = [c for c in rem if isinstance(c, dict) and "date" in c and datetime.fromisoformat(c["date"]) > cut] if isinstance(rem, list) else []
    print(f"{rid}")
    print(f"   cached revisions: {[(r['commit'][:10], r['last_modified_utc'][:19]) for r in loc] or 'none'}")
    if isinstance(rem, list):
        print(f"   latest Hub commit: {rem[0]['date'][:19] if rem else '?'}  | commits after 14 Sept: {len(late)}"
              + (f"  <-- {[(c['commit'][:10], c['date'][:10], c['title']) for c in late]}" if late else ""))
    else:
        print(f"   Hub history: {rem}")
(OUT_DIR / "rb1_forensics.json").write_text(json.dumps(FOR, indent=1, default=str))
print("\nforensics saved to rb1_forensics.json")

key packages:
  datasets                   5.1.0              2026-11-02T04:08:03.231079+00:00  <-- installed/changed after 14 Sept
  fsspec                     2026.7.0           2026-11-02T04:07:14.516058+00:00  <-- installed/changed after 14 Sept
  numpy                      2.4.6              2026-11-02T04:07:14.093857+00:00  <-- installed/changed after 14 Sept
  nvidia-cublas-cu12         12.8.4.1           2026-11-02T04:05:48.394545+00:00  <-- installed/changed after 14 Sept
  nvidia-cuda-nvrtc-cu12     12.8.93            2026-11-02T04:05:44.532976+00:00  <-- installed/changed after 14 Sept
  nvidia-cuda-runtime-cu12   12.8.90            2026-11-02T04:05:43.543993+00:00  <-- installed/changed after 14 Sept
  nvidia-cudnn-cu12          9.10.2.21          2026-11-02T04:06:00.320909+00:00  <-- installed/changed after 14 Sept
  nvidia-cufft-cu12          11.3.3.83          2026-11-02T04:05:53.931157+00:00  <-- installed/changed after 14 Sept
  nvidia-nvjitlink-cu12      12.8.93      


timm/vit_small_patch16_rope_224.naver_in1k
   cached revisions: [('91eb2b2dbd', '2026-09-12T22:42:50')]
   latest Hub commit: 2025-07-02T15:41:13  | commits after 14 Sept: 0
timm/vit_base_patch16_rope_224.naver_in1k
   cached revisions: [('7845c68231', '2026-09-09T07:56:44')]
   latest Hub commit: 2025-07-02T15:40:01  | commits after 14 Sept: 0
timm/vit_base_patch16_dinov3.lvd1689m
   cached revisions: [('c6a5fb7d12', '2026-09-09T08:02:15')]
   latest Hub commit: 2025-09-24T22:41:56  | commits after 14 Sept: 0
benjamin-paine/imagenet-1k-256x256
   cached revisions: [('1bd0400450', '2026-09-09T07:36:06')]
   latest Hub commit: 2024-09-15T11:15:04  | commits after 14 Sept: 0

forensics saved to rb1_forensics.json


## Anchors
September (W5) values come from the repo's committed result files. RB-0 values are typed below at
full precision (from `w8_partA_*.json`) and checked against the RB-0 folder when it is found.

In [5]:
RES_ROOT = repo.parents[1] / "results" / "runs" / "imagenet1k_val" / "rope_interventions"
W5 = {}
for key in MODEL_NAMES:
    hits = sorted(RES_ROOT.glob(f"w5/w5_partA_{key}_*.json"))
    if not hits:
        gate("committed W5 file found", key, False, "missing"); continue
    c = json.load(open(hits[0]))["conditions"]
    W5[key] = {"baseline": c["baseline"]["ssdc"], "floor": c["floor"]["ssdc"], "Cref": c["Cref_prefix5"]["ssdc"]}
HEAD_W5 = {"small": (0.371434, 0.244217), "naver": (0.469422, 0.329663), "DINOv3": (0.948654, 0.442640)}
for key, (b4, c11) in HEAD_W5.items():
    if key in W5:
        gate("committed W5 file matches its headline values", key,
             abs(W5[key]["baseline"][4] - b4) < 1e-6 and abs(W5[key]["Cref"][11] - c11) < 1e-6)

RB0 = {
    "small": {
        "baseline": [0.016578371328500142, 0.00395960172172849, 0.04124018025894116, 0.16859474094367033, 0.3713740002085806, 0.40662082251490533, 0.5489240691401012, 0.5064005532218341, 0.46002794920552376, 0.4585050970469657, 0.4455475282553677, 0.40882081445458496],
        "floor": [0.016578371328500142, 0.01519492949780967, 0.008314211653128038, 0.012237652830778625, 0.011329560521931974, 0.015282514783585829, 0.017073999258734963, 0.015751402426616903, 0.015598518859184712, 0.015369927485402567, 0.01594396006578761, 0.014931848731288264],
        "Cref": [0.016578371328500142, 0.01519492949780967, 0.008314211653128038, 0.012237652830778625, 0.011329560521931974, 0.015282514783585829, 0.017073999258734963, 0.06479249663883876, 0.17449259255126665, 0.17682112083446647, 0.21653782642309194, 0.24409487444912628],
    },
    "naver": {
        "baseline": [0.026535466343308493, 0.043699186842275264, 0.1091637047056099, 0.26788438341554316, 0.46942270929953334, 0.571301827637807, 0.5371129475808915, 0.5339065229528429, 0.5747029357196607, 0.5433898595996087, 0.4822155404687344, 0.45514819071073037],
        "floor": [0.026535466343308493, 0.013624620133663239, 0.010839588339364804, 0.011854571299371886, 0.012864818911651489, 0.015182811687894397, 0.01749119709271766, 0.015989610289790096, 0.017997490184067134, 0.01909746052484506, 0.021162288836076446, 0.022397308992923307],
        "Cref": [0.026535466343308493, 0.013624620133663239, 0.010839588339364804, 0.011854571299371886, 0.012864818911651489, 0.015182811687894397, 0.01749119709271766, 0.1583403691567614, 0.25113060615090343, 0.27180261502695613, 0.2961522945182981, 0.3296631443404199],
    },
    "DINOv3": {
        "baseline": [0.028061419654428127, 0.03723281990023723, 0.8416442362253422, 0.9360012704698906, 0.9486539607489823, 0.9314896120679399, 0.9404113062112893, 0.9414559726505077, 0.9325356727702198, 0.9300058708112987, 0.9305605942919077, 0.8522157092265887],
        "floor": [0.028061419654428127, 0.021471794619951495, 0.023721131535796264, 0.019373092589102255, 0.017711414187650976, 0.015312429216180267, 0.015525870121275985, 0.014515644619583141, 0.016405283766901212, 0.018909589709228538, 0.019979328858547377, 0.02528685315219093],
        "Cref": [0.028061419654428127, 0.021471794619951495, 0.023721131535796264, 0.019373092589102255, 0.017711414187650976, 0.015312429216180267, 0.015525870121275985, 0.4553550520783082, 0.4616016305073708, 0.4758900092424254, 0.5543148704736773, 0.4426402755470917],
    },
}
w8 = Path(W8_DIR).expanduser() if W8_DIR else None
if w8 is None:
    for cand in [Path.cwd() / "w8_output", Path.cwd().parent / "w8" / "w8_output", Path.cwd().parent / "w8_output"]:
        if (cand / "w8_partA_small.json").exists(): w8 = cand; break
if w8 and (w8 / "w8_partA_small.json").exists():
    dev = 0.0
    for key in RB0:
        c = json.load(open(w8 / f"w8_partA_{key}.json"))["conditions"]
        for cond in CONDITIONS:
            dev = max(dev, max(abs(a - b) for a, b in zip(c[cond]["ssdc"][:12], RB0[key][cond])))
    gate("typed RB-0 anchors equal the w8_partA files", "anchors", dev == 0.0, f"{w8} max |delta| {dev:.1e}")
else:
    gate("typed RB-0 anchors equal the w8_partA files", "anchors", True, "RB-0 folder not found; set RB1_W8_DIR", skip=True)

  [PASS] small: committed W5 file matches its headline values


  [PASS] naver: committed W5 file matches its headline values


  [PASS] DINOv3: committed W5 file matches its headline values


  [PASS] anchors: typed RB-0 anchors equal the w8_partA files: /workspace/NRDL/experiments/w8/w8_output max |delta| 0.0e+00


## The worker
One setting = one fresh Python process running this script. It loads each model with the repo's
`load_rope(input_size=224)`, runs W5's warm-up forward (W5's `forward_grid`), then measures baseline,
floor and Cref with `evaluate_ssdc` exactly as W5's `measure()` did. Afterwards (so the measurement
order matches W5) it hashes the images and weights and saves the patch embeddings of the first 2
images (enough to see whether two settings computed the first layer bit for bit the same).

In [6]:
WORKER = r'''
import os, sys, json, time, hashlib, argparse
ap = argparse.ArgumentParser(); ap.add_argument("--cfg", required=True); args = ap.parse_args()
CFG = json.load(open(args.cfg))
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
# flags are set before the first cuDNN call
torch.backends.cudnn.benchmark = False
if "cudnn_allow_tf32" in CFG["flags"]: torch.backends.cudnn.allow_tf32 = bool(CFG["flags"]["cudnn_allow_tf32"])
if "cudnn_deterministic" in CFG["flags"]: torch.backends.cudnn.deterministic = bool(CFG["flags"]["cudnn_deterministic"])
sys.path.insert(0, CFG["repo"])
from metrics.ssdc import evaluate_ssdc
from main.load_models import get_vit_blocks, get_block_attention, get_patch_embed_conv, load_rope
from main.prep_data import prep_data
from experiments.common import load_imagenet
TEST = CFG["test_mode"]; DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT = {"setting": CFG["name"], "flags": CFG["flags"], "models": {}, "env": {}}
t0 = time.time()
e = OUT["env"]
e["torch"] = torch.__version__; e["cuda"] = torch.version.cuda
e["cudnn"] = torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None
e["cudnn_allow_tf32"] = torch.backends.cudnn.allow_tf32; e["matmul_allow_tf32"] = torch.backends.cuda.matmul.allow_tf32
e["cudnn_deterministic"] = torch.backends.cudnn.deterministic; e["cudnn_benchmark"] = torch.backends.cudnn.benchmark
e["float32_matmul_precision"] = torch.get_float32_matmul_precision()
e["cuda_visible_devices"] = os.environ.get("CUDA_VISIBLE_DEVICES")
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    e["gpu"] = p.name; e["gpu_uuid"] = str(getattr(p, "uuid", "")); e["sm_count"] = p.multi_processor_count
    free, total = torch.cuda.mem_get_info(); e["free_gb_at_start"] = free / 1e9; e["total_gb"] = total / 1e9

# ---- W5 helpers, verbatim ---------------------------------------------------------------
def find_rope(model):
    for n, m in model.named_modules():
        if callable(getattr(m, "get_embed", None)): return n, m
    return None, None
def identity_like(emb):
    s, c = emb.chunk(2, -1); return torch.cat([torch.zeros_like(s), torch.ones_like(c)], -1)
class RopeGlobal:
    def __init__(s, model, tf): s.name, s.mod = find_rope(model); s.tf = tf; s.handles = []
    def __enter__(s):
        s.orig = s.mod.get_embed; o, t = s.orig, s.tf
        s.mod.get_embed = (lambda _o=o, _t=t: (lambda *a, **k: _t(_o(*a, **k))))()
        s.handles = [1]; return s
    def __exit__(s, *e): s.mod.get_embed = s.orig; s.handles = []
class RopeComposite:
    def __init__(s, model, source, segs):
        s.blocks = get_vit_blocks(model, source); s.source = source; s.segs = [(list(b), t) for b, t in segs]; s.handles = []
    def __enter__(s):
        def mk(tf):
            def pre(mod, args, kw):
                r = kw.get("rope", None)
                if r is not None: kw = dict(kw); kw["rope"] = tf(r)
                return args, kw
            return pre
        for blocks, tf in s.segs:
            for i in blocks:
                s.handles.append(get_block_attention(s.blocks[i], s.source).register_forward_pre_hook(mk(tf), with_kwargs=True))
        return s
    def __exit__(s, *e):
        for h in s.handles: h.remove()
        s.handles = []
class NullCtx:
    handles = []
    def __enter__(s): return s
    def __exit__(s, *e): return False

# ---- RB-1 addition: patch embedding computed in float64, rounded to fp32 ------------------
class ExactPatch:
    def __init__(s, conv): s.conv = conv
    def __enter__(s):
        c = s.conv
        def fwd(x, c=c):
            y = F.conv2d(x.double(), c.weight.double(), None if c.bias is None else c.bias.double(),
                         c.stride, c.padding, c.dilation, c.groups)
            return y.to(x.dtype)
        c.forward = fwd; return s
    def __exit__(s, *e): s.conv.__dict__.pop("forward", None)

# ---- data ------------------------------------------------------------------------------
if TEST:
    from PIL import Image as _PIL
    class _Fake:
        def __iter__(self):
            rng = np.random.default_rng(1)
            for i in range(400):
                yield {"image": _PIL.fromarray(rng.integers(0, 256, size=(256, 256, 3), dtype=np.uint8)), "label": i % 10}
    DATASET = _Fake()
else:
    DATASET = load_imagenet(split="validation", streaming=True, dataset_id=CFG["dataset"], fallback_id=None)

def load(key):
    if TEST:
        import timm
        torch.manual_seed(1234 + len(key))   # same tiny random weights in every setting
        nh = 6 if key == "small" else 12
        m = timm.create_model(CFG["model_names"][key].split(".")[0], pretrained=False, dynamic_img_size=True,
                              embed_dim=96 if key == "small" else 192, num_heads=nh,
                              num_classes=0 if key == "DINOv3" else 10).to(DEVICE).eval()
        with torch.no_grad():
            for blk in m.blocks:
                for g in ("gamma_1", "gamma_2"):
                    t = getattr(blk, g, None)
                    if isinstance(t, torch.Tensor): t.fill_(1.0)
        cfg = timm.data.resolve_model_data_config(m); cfg["input_size"] = (3, CFG["input_size"], CFG["input_size"])
        return m, timm.data.create_transform(**cfg, is_training=False), "timm"
    return load_rope(model_name=CFG["model_names"][key], device=DEVICE, half=False, input_size=CFG["input_size"])

N, B, SEED = CFG["n_images"], CFG["batch_size"], CFG["seed"]
for key in CFG["models"]:
    tm = time.time()
    model, processor, source = load(key)
    model.eval().float()
    conv = get_patch_embed_conv(model, source)
    gctx = ExactPatch(conv) if CFG["flags"].get("exact_patch") else NullCtx()
    rec = {"conditions": {}}
    with gctx:
        # W5 forward_grid: one forward of the first batch before anything is measured
        dl = prep_data(DATASET, processor, source, corruption_type=None, number_images=B, batch_size=B, half=False, num_workers=0)
        with torch.no_grad():
            for im, _ in dl:
                model(im.to(DEVICE)); break
        def measure(ctx):
            torch.manual_seed(SEED); np.random.seed(SEED % (2 ** 31))
            with ctx:
                scores, _ = evaluate_ssdc(model, processor, DATASET, source, RPI=True, number_images=N, batch_size=B,
                                          metric=CFG["metric"], half=False, num_workers=0)
            return [float(v) for v in scores]
        rec["conditions"]["baseline"] = measure(NullCtx())
        rec["conditions"]["floor"] = measure(RopeGlobal(model, identity_like))
        rec["conditions"]["Cref"] = measure(RopeComposite(model, source, [(CFG["window"], identity_like)]))
        # after measuring: image hash over all N images, patch embeddings of the first 16
        dl = prep_data(DATASET, processor, source, corruption_type=None, number_images=N, batch_size=B, half=False, num_workers=0)
        h = hashlib.sha256(); first = None; n_seen = 0
        for im, lb in dl:
            h.update(im.contiguous().numpy().tobytes()); h.update(torch.as_tensor(lb).numpy().tobytes())
            n_seen += int(im.shape[0])
            if first is None: first = im[:2].clone()
        rec["image_sha256"] = h.hexdigest(); rec["n_images_hashed"] = n_seen
        with torch.no_grad():
            pe = conv(first.to(DEVICE)).float().cpu()
        torch.save(pe, CFG["out_prefix"] + f"_{key}_patch2.pt")
    hw = hashlib.sha256()
    for k_, v_ in sorted(model.state_dict().items()):
        hw.update(k_.encode()); hw.update(v_.detach().float().cpu().contiguous().numpy().tobytes())
    rec["weights_sha256"] = hw.hexdigest()
    rec["runtime_s"] = round(time.time() - tm, 1)
    OUT["models"][key] = rec
    print(f"  {CFG['name']:20s} {key:7s} b4 baseline {rec['conditions']['baseline'][4]:.9f}  Cref b11 {rec['conditions']['Cref'][11]:.9f}  {rec['runtime_s']}s", flush=True)
    del model
    if DEVICE == "cuda": torch.cuda.empty_cache()
OUT["runtime_s"] = round(time.time() - t0, 1)
json.dump(OUT, open(CFG["out_prefix"] + ".json", "w"), indent=1)
print("WORKER DONE", flush=True)
'''
WORKER_PATH = OUT_DIR / "rb1_worker.py"
WORKER_PATH.write_text(WORKER)
print("worker written:", WORKER_PATH)

HOG = r'''
import sys, time, torch
n = int(float(sys.argv[1]) * 1e9) // 4
x = torch.empty(n, dtype=torch.float32, device="cuda"); x.fill_(1.0); torch.cuda.synchronize()
open(sys.argv[2], "w").write("ready"); time.sleep(float(sys.argv[3]))
'''
HOG_PATH = OUT_DIR / "rb1_hog.py"
HOG_PATH.write_text(HOG)

worker written: rb1_output/rb1_worker.py


229

## Run the settings
Each setting runs in a fresh process. The memory-hog setting first starts a separate process that
holds the memory, waits until it is allocated, runs the worker on the same GPU, then stops the hog.
With `RB1_EXTRA_GPUS=1`, the default setting is also run on every other visible GPU.

In [7]:
def settings_to_run():
    out = list(SETTINGS)
    if EXTRA_GPU_RUNS and not TEST_MODE and len(GPU_INDICES) > 1:
        for g in GPU_INDICES[1:]:
            out.append((f"gpu_{g}", {"gpu_index": g}, ["small", "naver"]))
    if ONLY:
        out = [s for s in out if s[0] in ONLY]
    return out

def run_setting(name, flags, models):
    prefix = str(OUT_DIR / f"rb1_{name}")
    done = Path(prefix + ".json")
    if done.exists() and not ONLY:
        print(f"{name}: already done, loading"); return json.loads(done.read_text())
    cfg = {"name": name, "flags": flags, "models": models, "repo": str(repo), "dataset": SSDC_DATASET,
           "n_images": N_IMAGES, "batch_size": BATCH_SIZE, "seed": TORCH_SEED, "metric": METRIC,
           "input_size": INPUT_SIZE, "window": WINDOW, "model_names": MODEL_NAMES, "test_mode": TEST_MODE,
           "out_prefix": prefix}
    cfg_path = Path(prefix + "_cfg.json"); cfg_path.write_text(json.dumps(cfg))
    env = dict(os.environ)
    gpu = flags.get("gpu_index", GPU_INDICES[0] if GPU_INDICES else 0)
    if not TEST_MODE:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    log_path = Path(prefix + "_cudnn.log")
    if CUDNN_LOGGING and not TEST_MODE:
        env.update({"CUDNN_LOGLEVEL_DBG": "3", "CUDNN_LOGDEST_DBG": str(log_path),
                    "CUDNN_LOGINFO_DBG": "1", "CUDNN_FRONTEND_LOG_INFO": "1",
                    "CUDNN_FRONTEND_LOG_FILE": str(log_path) + ".frontend"})
    hog = None; hog_info = None
    try:
        if flags.get("hog_gb"):
            flag_file = Path(prefix + "_hog_ready")
            flag_file.unlink(missing_ok=True)
            henv = dict(os.environ); henv["CUDA_VISIBLE_DEVICES"] = str(gpu)
            hog = subprocess.Popen([sys.executable, str(HOG_PATH), str(flags["hog_gb"]), str(flag_file), "3600"], env=henv)
            t = time.time()
            while not flag_file.exists():
                if hog.poll() is not None: raise RuntimeError(f"hog process exited with {hog.returncode}")
                if time.time() - t > 300: raise RuntimeError("hog did not allocate within 300 s")
                time.sleep(1)
            hog_info = sh(f"nvidia-smi -i {gpu} --query-gpu=memory.used,memory.total --format=csv,noheader")
            print(f"{name}: hog holding {flags['hog_gb']} GB on GPU {gpu}; GPU memory now {hog_info.strip()}")
        t = time.time()
        r = subprocess.run([sys.executable, str(WORKER_PATH), "--cfg", str(cfg_path)], env=env,
                           capture_output=True, text=True, timeout=3600)
        Path(prefix + "_stdout.txt").write_text(r.stdout + "\n--- stderr ---\n" + r.stderr)
        print(r.stdout.strip())
        if r.returncode != 0 or not done.exists():
            print(r.stderr[-3000:])
            gate("setting ran to completion", name, False, f"exit {r.returncode}"); return None
    finally:
        if hog is not None:
            hog.terminate()
            try: hog.wait(30)
            except Exception: hog.kill()
    out = json.loads(done.read_text())
    out["hog_gpu_memory"] = hog_info
    # cuDNN engine lines, if logging produced any
    lines = []
    for lp in (log_path, Path(str(log_path) + ".frontend")):
        if lp.exists():
            sz = lp.stat().st_size
            with open(lp, errors="ignore") as fh:
                for ln in fh:
                    if re.search(r"engine|knob|algo", ln, re.I):
                        lines.append(ln.strip()[:240])
            if sz > 5e6:   # keep the archive small
                with open(lp, errors="ignore") as fh: head = fh.read(2_000_000)
                lp.write_text(head + f"\n... truncated from {sz} bytes ...\n")
    out["cudnn_engine_lines"] = sorted(set(lines))[:400]
    done.write_text(json.dumps(out, indent=1))
    gate("setting ran to completion", name, True, f"{time.time() - t:.0f}s")
    return out

RUNS = {}
T0 = time.time()
for name, flags, models in settings_to_run():
    print("\n" + "=" * 78 + f"\n{name}  flags={flags}  models={models}\n" + "=" * 78, flush=True)
    try:
        RUNS[name] = run_setting(name, flags, models)
    except Exception as ex:
        traceback.print_exc(); gate("setting ran to completion", name, False, repr(ex)[:200])
for f in OUT_DIR.glob("rb1_*.json"):   # pick up settings finished in an earlier session
    n = f.stem[4:]
    if n not in RUNS and not n.endswith("_cfg") and n not in ("forensics",) and not n.startswith("gates") and not n.startswith("all"):
        try:
            d = json.loads(f.read_text())
            if "models" in d and "setting" in d: RUNS[n] = d
        except Exception:
            pass
RUNS = {k: v for k, v in RUNS.items() if v}
print(f"\n{len(RUNS)} settings available, {(time.time() - T0) / 60:.1f} min this session")


default_a  flags={}  models=['small', 'naver']


default_a: already done, loading

default_b  flags={}  models=['small', 'naver']


default_b: already done, loading

w8_order  flags={}  models=['naver', 'DINOv3', 'small']


w8_order: already done, loading

tf32_off  flags={'cudnn_allow_tf32': False}  models=['small', 'naver']


tf32_off: already done, loading

cudnn_deterministic  flags={'cudnn_deterministic': True}  models=['small', 'naver']


cudnn_deterministic: already done, loading

hog_27GB  flags={'hog_gb': 27.9}  models=['small', 'naver']


hog_27GB: already done, loading

exact_patch  flags={'exact_patch': True}  models=['small', 'naver']


exact_patch: already done, loading

8 settings available, 0.0 min this session


## Analysis
For every setting, model and condition: the largest difference over the 12 block inputs against
September (W5) and against RB-0. **"0"** means bit-identical, as W2-W6 were with each other.

In [8]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 80)

def maxdev(a, b): return max(abs(x - y) for x, y in zip(a, b))
def fmt(d):
    if d is None: return "-"
    return "0" if d == 0.0 else f"{d:.1e}"

rows = []
for name, R in RUNS.items():
    for key, rec in R["models"].items():
        for cond in CONDITIONS:
            v = rec["conditions"][cond]
            d5 = maxdev(v, W5[key][cond]) if key in W5 and not TEST_MODE else None
            d0 = maxdev(v, RB0[key][cond]) if key in RB0 and not TEST_MODE else None
            rows.append({"setting": name, "model": key, "condition": cond, "vs_W5": d5, "vs_RB0": d0,
                         "b0_vs_W5": (v[0] - W5[key][cond][0]) if key in W5 and not TEST_MODE else None,
                         "b11_vs_W5": (v[11] - W5[key][cond][11]) if key in W5 and not TEST_MODE else None})
DF = pd.DataFrame(rows)
if len(DF):
    show = DF.copy()
    for c in ("vs_W5", "vs_RB0"): show[c] = show[c].map(fmt)
    for c in ("b0_vs_W5", "b11_vs_W5"): show[c] = show[c].map(lambda x: "-" if x is None or pd.isna(x) else f"{x:+.1e}")
    print("max |difference| over the 12 block inputs\n")
    print(show.to_string(index=False))

def summary(key, ref):
    out = {}
    for name in RUNS:
        sub = DF[(DF.setting == name) & (DF.model == key)]
        col = "vs_W5" if ref == "W5" else "vs_RB0"
        if len(sub) and sub[col].notna().all(): out[name] = float(sub[col].max())
    return out
SUM = {key: {"vs_W5": summary(key, "W5"), "vs_RB0": summary(key, "RB0")} for key in MODEL_NAMES}
print("\nper setting, worst case over all three conditions:")
for key in MODEL_NAMES:
    if SUM[key]["vs_W5"]:
        print(f"  {key:7s} vs September: " + "  ".join(f"{n}={fmt(d)}" for n, d in SUM[key]["vs_W5"].items()))
        print(f"  {key:7s} vs RB-0     : " + "  ".join(f"{n}={fmt(d)}" for n, d in SUM[key]["vs_RB0"].items()))

max |difference| over the 12 block inputs

            setting  model condition   vs_W5  vs_RB0 b0_vs_W5 b11_vs_W5
          default_a  small  baseline 4.1e-04       0 -1.3e-06  -4.1e-04
          default_a  small     floor 8.3e-05       0 -1.3e-06  +1.5e-05
          default_a  small      Cref 1.2e-04       0 -1.3e-06  -1.2e-04
          default_a  naver  baseline 2.4e-07       0 -6.0e-08  +2.0e-07
          default_a  naver     floor 5.6e-07       0 -6.0e-08  -5.6e-07
          default_a  naver      Cref 3.6e-07       0 -6.0e-08  +2.5e-07
          default_b  small  baseline 4.1e-04       0 -1.3e-06  -4.1e-04
          default_b  small     floor 8.3e-05       0 -1.3e-06  +1.5e-05
          default_b  small      Cref 1.2e-04       0 -1.3e-06  -1.2e-04
          default_b  naver  baseline 2.4e-07       0 -6.0e-08  +2.0e-07
          default_b  naver     floor 5.6e-07       0 -6.0e-08  -5.6e-07
          default_b  naver      Cref 3.6e-07       0 -6.0e-08  +2.5e-07
           w8_order  

In [9]:
# inputs, weights, patch embeddings, cuDNN engines
import torch   # CPU only from here on
for key in MODEL_NAMES:
    img = {n: R["models"][key]["image_sha256"] for n, R in RUNS.items() if key in R["models"]}
    wts = {n: R["models"][key]["weights_sha256"] for n, R in RUNS.items() if key in R["models"]}
    if img:
        gate("images identical in every setting", key, len(set(img.values())) == 1, f"sha256 {next(iter(img.values()))[:16]}")
        gate("weights identical in every setting", key, len(set(wts.values())) == 1, f"sha256 {next(iter(wts.values()))[:16]}")

PE = {}
for name in RUNS:
    for key in MODEL_NAMES:
        f = OUT_DIR / f"rb1_{name}_{key}_patch2.pt"
        if f.exists(): PE[(name, key)] = torch.load(f, map_location="cpu")
def rel(a, b): return float((a - b).abs().max() / b.abs().max().clamp_min(1e-12))
PEROWS = []
for key in MODEL_NAMES:
    ref = PE.get(("exact_patch", key))
    for name in RUNS:
        x = PE.get((name, key))
        if x is None or ref is None: continue
        PEROWS.append({"model": key, "setting": name, "patch_embed_rel_diff_vs_exact": rel(x, ref),
                       "bit_identical_to_default_a": bool(("default_a", key) in PE and torch.equal(x, PE[("default_a", key)]))})
PEDF = pd.DataFrame(PEROWS)
if len(PEDF):
    print("\npatch embeddings of the first 2 images: largest difference from the float64 reference, relative to the largest value")
    print(PEDF.to_string(index=False))
if ("exact_patch", "small") in PE and ("default_a", "small") in PE:
    gate("exact patch embedding computes the same convolution (relative difference < 1e-2)", "exact_patch",
         rel(PE[("default_a", "small")], PE[("exact_patch", "small")]) < 1e-2)

# SSDC moved by first-layer rounding alone (default vs exact, default vs tf32_off)
SENS = {}
for key in MODEL_NAMES:
    if not all(n in RUNS and key in RUNS[n]["models"] for n in ("default_a", "exact_patch")): continue
    s = {}
    for other in ("exact_patch", "tf32_off"):
        if other in RUNS and key in RUNS[other]["models"]:
            d = [maxdev(RUNS["default_a"]["models"][key]["conditions"][c], RUNS[other]["models"][key]["conditions"][c]) for c in CONDITIONS]
            s[f"default_vs_{other}"] = max(d)
    if key in W5 and not TEST_MODE:
        s["W5_vs_RB0"] = max(maxdev(W5[key][c], RB0[key][c]) for c in CONDITIONS)
    SENS[key] = s
print("\nfirst-layer rounding effect on SSDC (worst block, worst condition), next to the September-to-November drift:")
for key, s in SENS.items():
    print(f"  {key:7s} " + "  ".join(f"{k}={fmt(v)}" for k, v in s.items()))

ENG = {n: R.get("cudnn_engine_lines", []) for n, R in RUNS.items()}
if any(ENG.values()):
    base = set(ENG.get("default_a", []))
    print("\ncuDNN log lines mentioning an engine/algorithm, compared with default_a:")
    for n, L in ENG.items():
        print(f"  {n:20s} {len(L):4d} lines | same as default_a: {set(L) == base}")
else:
    print("\n(no cuDNN engine lines were logged; the verdict below does not depend on them)")

  [PASS] small: images identical in every setting: sha256 e6d5e9be840755fa


  [PASS] small: weights identical in every setting: sha256 c56cefca2f0fccb7


  [PASS] naver: images identical in every setting: sha256 e6d5e9be840755fa


  [PASS] naver: weights identical in every setting: sha256 c372708a448440ee


  [PASS] DINOv3: images identical in every setting: sha256 48d923c6f79a4e1e


  [PASS] DINOv3: weights identical in every setting: sha256 96c6d59af51c88aa



patch embeddings of the first 2 images: largest difference from the float64 reference, relative to the largest value
model             setting  patch_embed_rel_diff_vs_exact  bit_identical_to_default_a
small           default_a                   1.032782e-06                        True
small           default_b                   1.032782e-06                        True
small            w8_order                   1.032782e-06                        True
small            tf32_off                   1.032782e-06                        True
small cudnn_deterministic                   1.032782e-06                        True
small            hog_27GB                   1.032782e-06                        True
small         exact_patch                   0.000000e+00                       False
small            hog_58GB                   1.032782e-06                        True
naver           default_a                   8.044624e-07                        True
naver           default_b       


first-layer rounding effect on SSDC (worst block, worst condition), next to the September-to-November drift:
  small   default_vs_exact_patch=3.3e-07  default_vs_tf32_off=0  W5_vs_RB0=4.1e-04
  naver   default_vs_exact_patch=9.4e-05  default_vs_tf32_off=9.4e-05  W5_vs_RB0=5.6e-07

(no cuDNN engine lines were logged; the verdict below does not depend on them)


## Verdict
The rules are fixed here, before anyone sees the numbers.

In [10]:
def exact_settings(key, ref):
    return [n for n, d in SUM.get(key, {}).get(ref, {}).items() if d == 0.0]
def near_settings(key, ref, tol=1e-9):
    return [n for n, d in SUM.get(key, {}).get(ref, {}).items() if d <= tol]

VERDICT = {}
if not TEST_MODE and "small" in SUM:
    same_now = SUM["small"]["vs_RB0"]
    VERDICT["settings_reproducing_RB0_exactly"] = exact_settings("small", "vs_RB0")
    VERDICT["settings_reproducing_September_exactly"] = exact_settings("small", "vs_W5")
    VERDICT["settings_within_1e-9_of_September"] = near_settings("small", "vs_W5")
    da, db = RUNS.get("default_a"), RUNS.get("default_b")
    if da and db:
        rep = max(maxdev(da["models"][k]["conditions"][c], db["models"][k]["conditions"][c]) for k in da["models"] for c in CONDITIONS)
        VERDICT["default_repeatability"] = rep
        gate("current environment is repeatable (default_a equals default_b)", "env", rep == 0.0, f"max |delta| {rep:.1e}")
    gate("the current environment still reproduces RB-0 in at least one setting", "env",
         bool(VERDICT["settings_reproducing_RB0_exactly"]), str(VERDICT["settings_reproducing_RB0_exactly"]))
    if "naver" in SUM and "default_a" in SUM["naver"]["vs_W5"]:
        d = SUM["naver"]["vs_W5"]["default_a"]
        gate("naver stays within RB-0's 1.1e-6 of September", "naver", d <= 1.1e-6 + 1e-9, fmt(d))

    hog = [n for n in VERDICT["settings_reproducing_September_exactly"] if n.startswith("hog")]
    other = [n for n in VERDICT["settings_reproducing_September_exactly"] if not n.startswith("hog")]
    sens = SENS.get("small", {}).get("default_vs_exact_patch")
    drift = SENS.get("small", {}).get("W5_vs_RB0")
    changed = [r["name"] for r in FOR.get("packages_changed_after_cutoff", []) if r["name"] in KEY_PKGS]
    hub_late = [rid for rid, rem in FOR.get("hf_remote", {}).items()
                if isinstance(rem, list) and any(datetime.fromisoformat(c["date"]) > cut for c in rem)]
    if hog:
        case = ("A", "Free GPU memory decides which convolution kernel ViT-S's patch embedding uses. With the "
                "September co-tenant's memory taken, today's machine reproduces September bit for bit. The drift came "
                "from that job no longer sharing the GPU, not from our code, data or weights.")
    elif other:
        case = ("B", f"Setting(s) {other} reproduce September bit for bit, so September ran with that behaviour. "
                "Today's default differs only in that one respect.")
    elif sens is not None and drift is not None and sens >= 0.1 * drift:
        case = ("C", "No setting brings back September exactly, but rounding in the first layer alone moves ViT-S's SSDC "
                f"by {fmt(sens)} against a drift of {fmt(drift)}, so a change in which TF32 convolution kernel is used is "
                "enough to explain the drift. The forensics list what changed on the machine: "
                f"packages {changed or 'none flagged'}, Hub commits after 14 Sept {hub_late or 'none'}.")
    else:
        case = ("D", "No setting brings back September, and first-layer rounding is too small to explain the drift. "
                f"Look at the inputs: packages changed after 14 Sept {changed or 'none flagged'}, "
                f"Hub commits after 14 Sept {hub_late or 'none'}.")
    VERDICT["case"], VERDICT["explanation"] = case
    if VERDICT.get("default_repeatability", 0.0) != 0.0:
        VERDICT["recommendation"] = ("The current machine is not repeatable run to run. For RB-2 to RB-5, set "
                                     "torch.backends.cudnn.deterministic = True if the cudnn_deterministic setting is stable, and "
                                     "gate every run against a reference measured in the same session.")
    else:
        VERDICT["recommendation"] = ("Keep PyTorch's defaults for RB-2 to RB-5 so every number stays comparable with W2 to RB-0. "
                                     "Gate new runs against RB-0 (same machine, bit for bit), never subtract a September ViT-S "
                                     "value from a November one, and record cuDNN, driver and free GPU memory in every output.")
    print(f"CASE {case[0]}: {case[1]}\n")
    print("settings reproducing RB-0 exactly      :", VERDICT["settings_reproducing_RB0_exactly"])
    print("settings reproducing September exactly :", VERDICT["settings_reproducing_September_exactly"])
    print("settings within 1e-9 of September      :", VERDICT["settings_within_1e-9_of_September"])
    print("\nRECOMMENDATION:", VERDICT["recommendation"])
else:
    print("test mode: verdict skipped")

  [PASS] env: current environment is repeatable (default_a equals default_b): max |delta| 0.0e+00


  [PASS] env: the current environment still reproduces RB-0 in at least one setting: ['default_a', 'default_b', 'w8_order', 'tf32_off', 'cudnn_deterministic', 'hog_27GB', 'hog_58GB']


  [PASS] naver: naver stays within RB-0's 1.1e-6 of September: 5.6e-07


CASE D: No setting brings back September, and first-layer rounding is too small to explain the drift. Look at the inputs: packages changed after 14 Sept ['datasets', 'fsspec', 'numpy', 'nvidia-cublas-cu12', 'nvidia-cuda-nvrtc-cu12', 'nvidia-cuda-runtime-cu12', 'nvidia-cudnn-cu12', 'nvidia-cufft-cu12', 'nvidia-nvjitlink-cu12', 'pillow', 'pyarrow', 'safetensors', 'scipy', 'timm', 'torch', 'torchvision', 'transformers', 'triton'], Hub commits after 14 Sept none.

settings reproducing RB-0 exactly      : ['default_a', 'default_b', 'w8_order', 'tf32_off', 'cudnn_deterministic', 'hog_27GB', 'hog_58GB']
settings reproducing September exactly : []
settings within 1e-9 of September      : []

RECOMMENDATION: Keep PyTorch's defaults for RB-2 to RB-5 so every number stays comparable with W2 to RB-0. Gate new runs against RB-0 (same machine, bit for bit), never subtract a September ViT-S value from a November one, and record cuDNN, driver and free GPU memory in every output.


## Gate report and export

In [11]:
_all = []
for f in sorted(OUT_DIR.glob("rb1_gates_session_*.json")):
    _all += json.loads(f.read_text())
_seen = {}
for g in _all: _seen[(g["scope"], g["gate"])] = g
gdf = pd.DataFrame(list(_seen.values()))
print(f"GATE REPORT ({len(set(g['session'] for g in _all))} session(s), latest result per gate)\n")
print(gdf.to_string(index=False) if len(gdf) else "(none)")
n_fail = int((gdf.status == "FAIL").sum()) if len(gdf) else 0
print(f"\n{len(gdf)} gates: {int((gdf.status == 'PASS').sum())} pass, {n_fail} fail, {int((gdf.status == 'SKIP').sum())} skipped")
print("ALL GATES PASSED" if n_fail == 0 else f"{n_fail} GATE(S) FAILED: send the whole rb1_output folder")

export = {"notebook": "RB-1", "generated_utc": RUN_STAMP, "test_mode": TEST_MODE, "repo_commit": _head,
          "protocol": {"dataset": SSDC_DATASET, "n_images": N_IMAGES, "batch": BATCH_SIZE, "seed": TORCH_SEED,
                       "metric": METRIC, "input_size": INPUT_SIZE, "window": WINDOW, "path": "W5 measure(): evaluate_ssdc, RPI, fp32"},
          "settings": {n: {"flags": R["flags"], "env": R["env"], "hog_gpu_memory": R.get("hog_gpu_memory")} for n, R in RUNS.items()},
          "comparison": DF.to_dict("records") if len(DF) else [], "summary": SUM, "patch_embed": PEROWS,
          "sensitivity": SENS, "verdict": VERDICT, "gates": gdf.to_dict("records") if len(gdf) else []}
(OUT_DIR / f"rb1_all_{RUN_STAMP}.json").write_text(json.dumps(export, indent=1, default=str))
if len(DF): DF.to_csv(OUT_DIR / "rb1_comparison.csv", index=False)
gdf.to_csv(OUT_DIR / "rb1_gate_report.csv", index=False)
archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print(f"\narchive: {archive} ({os.path.getsize(archive):,} bytes)")

GATE REPORT (3 session(s), latest result per gate)

                                                                            gate               scope status                                                                                            detail              session
                                         imported metric satisfies its unit test         repo-metric   PASS                                                                                     +1.000/-1.000 2026-11-04T00-06-44Z
                                   committed W5 file matches its headline values               small   PASS                                                                                                   2026-11-04T00-06-44Z
                                   committed W5 file matches its headline values               naver   PASS                                                                                                   2026-11-04T00-06-44Z
                                   commi


archive: /workspace/NRDL/experiments/rb1/rb1_output.zip (15,299,351 bytes)
